# Scale calculation from dataset statistics

Given the normalization type for each variable, compute the scale `s` such that
after `x_norm = s * (x - beta) / alpha`, all variables contribute roughly equally
to the loss.

**Formula:**

After normalization (before scaling), the effective std of `x_norm` is:

| norm_type  | alpha   | std(x_norm) |
|------------|---------|-------------|
| zscore     | σ       | 1.0         |
| std_only   | σ       | 1.0 (approx, since mean≠0) |
| none       | 1       | σ (raw std) |
| minmax     | max-min | σ/(max-min) |

We want `scale * std(x_norm) = target_std` for all variables.
So `scale = target_std / std(x_norm)`.

We use `target_std = 1.0` by default, but you can anchor to a specific
variable (e.g. `2t`) instead.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# Dataset statistics
# ─────────────────────────────────────────────────────────────────────────────
VAR_NAMES = [
    '10fg','10si','10u','10v','2d','2t','cbh','fog',
    'hcc','lcc','lsm','mcc','msl','q_850','sp','t_850',
    'tcc','tcw','tp','u_850','v_850','vis','w_850','z'
]

MAXIMUM = np.array([
    1.3997218e+02, 4.2214622e+01, 3.9473759e+01, 4.0713623e+01,
    3.0036462e+02, 3.1187714e+02, 2.9196381e+04, 1.0000000e+00,
    1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
    1.0518031e+05, 1.4661202e-02, 1.0570074e+05, 2.9572833e+02,
    1.0000000e+00, 9.5032677e+01, 7.7389650e-02, 5.1804878e+01,
    5.3662399e+01, 7.5001000e+04, 1.2886423e+02, 1.9807861e+04
])
MINIMUM = np.array([
     8.95893350e-02,  4.86747740e-05, -3.12859097e+01, -2.95091782e+01,
     2.25942673e+02,  2.35660233e+02,  1.05415258e+01,  0.00000000e+00,
     0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
     9.44786641e+04, -4.29647518e-09,  7.36904062e+04,  2.50080322e+02,
     0.00000000e+00,  2.14778781e-01, -7.81250037e-06, -4.02996368e+01,
    -4.07896729e+01,  1.77492447e+01, -1.17423256e+02, -9.10729553e+02
])
MEAN = np.array([
    7.16135120e+00, 3.72048378e+00,  7.37869620e-01,  4.60248023e-01,
    2.74782562e+02, 2.78345642e+02,  8.79989648e+03,  5.00954166e-02,
    3.82601112e-01, 4.37377602e-01,  8.05984497e-01,  3.02159548e-01,
    1.01162547e+05, 3.76774487e-03,  9.67466875e+04,  2.73641815e+02,
    6.53537095e-01, 1.30685177e+01,  1.09164524e-04,  3.48920178e+00,
    4.96566683e-01, 2.36682480e+04,  5.17917797e-02,  3.65718555e+03
])
STD = np.array([
    4.3299451e+00, 2.9582992e+00, 3.2886195e+00, 3.3199751e+00,
    7.9684067e+00, 8.5489264e+00, 6.6793906e+03, 2.0027253e-01,
    4.2964888e-01, 4.5027056e-01, 3.9544085e-01, 4.0948805e-01,
    1.1974965e+03, 1.9972378e-03, 4.5984819e+03, 6.6869164e+00,
    4.1893700e-01, 7.3980894e+00, 4.5502096e-04, 7.3267422e+00,
    7.5560985e+00, 2.1279119e+04, 2.2331698e+00, 3.7608040e+03
])

vi = {v: i for i, v in enumerate(VAR_NAMES)}  # name → dataset index

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Your normalization config (norm type per variable)
# ─────────────────────────────────────────────────────────────────────────────
NORM_CONFIG = {
    '10si':  'zscore',
    '10u':   'zscore',
    '10v':   'zscore',
    '2d':    'zscore',
    '2t':    'zscore',
    'cbh':   'zscore',
    'hcc':   'none',
    'lcc':   'none',
    'lsm':   'none',
    'mcc':   'none',
    'msl':   'zscore',
    'q_850': 'std_only',
    'sp':    'zscore',
    't_850': 'zscore',
    'tcc':   'none',
    'tcw':   'std_only',
    'tp':    'std_only',
    'u_850': 'zscore',
    'v_850': 'zscore',
    'w_850': 'std_only',
    'z':     'zscore',
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1: compute effective std AFTER normalization (before any scale)
#
# This is the std that the model/loss actually sees for each variable
# when scale=1.  We want all of these to be equal.
#
# norm_type   alpha         std(x_norm)
# ─────────────────────────────────────
# zscore      σ             1.0  (by definition)
# std_only    σ             √(1 + (μ/σ)²)  ≈ 1 when μ≈0, >1 when μ>>0
# minmax      max-min       σ / (max-min)
# none        1             σ  (raw std, can be huge)
# ─────────────────────────────────────────────────────────────────────────────
EPS = 1e-10

def effective_std(norm: str, mu: float, sig: float, mn: float, mx: float) -> float:
    """Std of x_norm when scale=1, computed analytically from dataset stats."""
    if norm == 'zscore':
        return 1.0                          # (x-mu)/sig always has std=1
    elif norm == 'std_only':
        # x/sig — std = sig/sig = 1, BUT mean = mu/sig ≠ 0
        # std of normalised values is still 1 (same as zscore for std)
        return 1.0
    elif norm == 'minmax':
        return sig / (mx - mn + EPS)        # (x-min)/(max-min)
    elif norm == 'none':
        return sig                           # identity — raw std
    else:
        raise ValueError(f"Unknown norm type: {norm}")


# ─────────────────────────────────────────────────────────────────────────────
# Step 2: compute scale
#
# Two methods — pick one:
#
# METHOD A — target_std = 1.0
#   scale = 1 / std(x_norm)
#   Result: all variables enter the loss with the same effective std.
#   Problem: for zscore/std_only, scale=1 always — no differentiation.
#            for none-normed cloud fractions (σ≈0.4), scale≈2.5 — too high.
#
# METHOD B — anchor to a reference variable (recommended)
#   Choose one variable whose scale=1 feels right (e.g. 2t).
#   scale_var = std(x_norm_ref) / std(x_norm_var)
#   Result: scales express relative importance vs the reference.
#   This is what AIFS does implicitly with their table.
# ─────────────────────────────────────────────────────────────────────────────

ANCHOR_VAR  = '2t'    # the variable whose scale will be 1.0
TARGET_STD  = 1.0     # target effective std for method A

rows = []
for vname, norm in NORM_CONFIG.items():
    i   = vi[vname]
    mu  = MEAN[i]
    sig = STD[i]
    mn  = MINIMUM[i]
    mx  = MAXIMUM[i]

    eff_std = effective_std(norm, mu, sig, mn, mx)

    # Method A: scale so that effective std = TARGET_STD
    scale_A = TARGET_STD / (eff_std + EPS)

    rows.append(dict(
        variable  = vname,
        norm      = norm,
        mu        = mu,
        sig       = sig,
        eff_std   = eff_std,
        scale_A   = scale_A,
    ))

df = pd.DataFrame(rows).set_index('variable')

# Method B: anchor to reference variable
anchor_eff_std = df.loc[ANCHOR_VAR, 'eff_std']
df['scale_B']  = anchor_eff_std / (df['eff_std'] + EPS)

# Round to clean values for the config
df['scale_A_rounded'] = df['scale_A'].apply(lambda x: float(f'{x:.4g}'))
df['scale_B_rounded'] = df['scale_B'].apply(lambda x: float(f'{x:.4g}'))

pd.set_option('display.float_format', '{:.5f}'.format)
pd.set_option('display.max_columns', 20)
df[['norm','sig','eff_std','scale_A','scale_B','scale_A_rounded','scale_B_rounded']]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Print ready-to-paste YAML
# Change USE_METHOD to 'A' or 'B'
# ─────────────────────────────────────────────────────────────────────────────
USE_METHOD = 'B'   # 'A' = unit variance for all, 'B' = relative to anchor var

scale_col = f'scale_{USE_METHOD}_rounded'

print(f"# Method {USE_METHOD}: {'target std=1.0' if USE_METHOD == 'A' else f'anchor={ANCHOR_VAR}'}")
print("normalizer:")
print("  _target_: dssml.data.helpers.normalizers.UnifiedNormalizer")
print("  default_norm_type: zscore")
print("  default_scale: 1.0")
print("  eps: 1.0e-6")
print("  overrides:")
for var, row in df.iterrows():
    norm  = row['norm']
    scale = row[scale_col]
    print(f"    {var}:")
    print(f"      norm_type: {norm}")
    print(f"      scale: {scale}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plot: effective std * scale for both methods — should be flat
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
vars_ = df.index.tolist()
x     = np.arange(len(vars_))

for ax, method in zip(axes, ['A', 'B']):
    effective = df['eff_std'].values * df[f'scale_{method}'].values
    ax.bar(x, effective, color='steelblue', alpha=0.8, edgecolor='none')
    ax.axhline(1.0, color='tomato', linewidth=1.0, linestyle='--', label='target=1')
    ax.set_xticks(x)
    ax.set_xticklabels(vars_, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('eff_std × scale')
    label = 'target std=1.0' if method == 'A' else f'anchor={ANCHOR_VAR}'
    ax.set_title(f'Method {method}: {label}', fontsize=10)
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Effective std × scale per variable (should be flat = 1)', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('scale_check.png', dpi=130, bbox_inches='tight')
plt.show()